# 07 — Phase 6: Correlation Analysis
**Football Players Stats 2024/25**

Correlation heatmaps and scatter plots to find relationships between key stats.

> Outfield players with 900+ minutes. GK analysis included separately.

## Setup

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

player_df = pd.read_csv('Dataset\player_clean.csv')

LEAGUE_COLORS = {
    'Premier League': '#3d195b', 'La Liga': '#ee8707',
    'Serie A': '#008fd7', 'Bundesliga': '#d3010c', 'Ligue 1': '#091c3e',
}
POSITION_COLORS = {'FW': '#e63946', 'MF': '#2a9d8f', 'DF': '#457b9d', 'GK': '#6a4c93'}

corr_df    = player_df[player_df['sufficient_minutes'] == True].copy()
outfield_df = corr_df[corr_df['primary_position'] != 'GK'].copy()
gk_df      = corr_df[corr_df['primary_position'] == 'GK'].copy()

print(f"Outfield: {len(outfield_df)} | GKs: {len(gk_df)}")

Outfield: 1457 | GKs: 119


In [2]:
# 6.1 Attacking Stats Correlation Heatmap
attacking_cols = ['goals', 'assists', 'xg', 'xag', 'shots', 'shots_on_target',
                  'key_passes', 'goal_creating_actions', 'shot_creating_actions',
                  'touches_opp_box', 'progressive_carries', 'progressive_passes']

attack_corr = outfield_df[attacking_cols].corr().round(2)
fig = px.imshow(attack_corr, text_auto=True, color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1, title='Attacking Stats Correlation Heatmap (Outfield Players)', aspect='auto')
fig.update_layout(height=580, coloraxis_colorbar=dict(title='Correlation'))
fig.show()

In [3]:
# 6.2 Defensive Stats Correlation Heatmap
defensive_cols = ['tackles', 'tackles_won', 'interceptions', 'tackles_interceptions',
                  'clearances', 'blocks', 'aerial_duels_won', 'aerial_duel_win_pct',
                  'errors_leading_to_shot', 'fouls_committed']

def_corr = outfield_df[defensive_cols].corr().round(2)
fig = px.imshow(def_corr, text_auto=True, color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1, title='Defensive Stats Correlation Heatmap (Outfield Players)', aspect='auto')
fig.update_layout(height=550, coloraxis_colorbar=dict(title='Correlation'))
fig.show()

In [4]:
# 6.3 Goals vs Shots on Target
fig = px.scatter(outfield_df, x='shots_on_target', y='goals', color='primary_position',
                 color_discrete_map=POSITION_COLORS,
                 hover_data=['player_name', 'club', 'league'], trendline='ols',
                 title='Goals vs Shots on Target — Does Volume = Goals? (2024/25)',
                 labels={'shots_on_target': 'Shots on Target', 'goals': 'Goals',
                         'primary_position': 'Position'}, opacity=0.6)
fig.update_layout(height=520, plot_bgcolor='white',
                  xaxis=dict(showgrid=True, gridcolor='#f0f0f0'),
                  yaxis=dict(showgrid=True, gridcolor='#f0f0f0'), legend_title='Position')
fig.show()

In [5]:
# 6.4 Pass Completion % vs Progressive Passes (min 200 passes)
pass_df = outfield_df[outfield_df['passes_attempted'] >= 200].copy()

fig = px.scatter(pass_df, x='pass_completion_pct', y='progressive_passes',
                 color='primary_position', color_discrete_map=POSITION_COLORS,
                 hover_data=['player_name', 'club', 'league'], trendline='ols',
                 title='Pass Completion % vs Progressive Passes (min 200 passes)',
                 labels={'pass_completion_pct': 'Pass Completion %',
                         'progressive_passes': 'Progressive Passes',
                         'primary_position': 'Position'}, opacity=0.6)
fig.update_layout(height=520, plot_bgcolor='white',
                  xaxis=dict(showgrid=True, gridcolor='#f0f0f0'),
                  yaxis=dict(showgrid=True, gridcolor='#f0f0f0'), legend_title='Position')
fig.show()

In [6]:
# 6.5 Key Passes vs Assists
fig = px.scatter(outfield_df, x='key_passes', y='assists', color='primary_position',
                 color_discrete_map=POSITION_COLORS,
                 hover_data=['player_name', 'club', 'league', 'key_passes', 'assists'],
                 trendline='ols', title='Key Passes vs Assists — Do Key Passes Convert? (2024/25)',
                 labels={'key_passes': 'Key Passes', 'assists': 'Assists',
                         'primary_position': 'Position'}, opacity=0.6)
fig.update_layout(height=520, plot_bgcolor='white',
                  xaxis=dict(showgrid=True, gridcolor='#f0f0f0'),
                  yaxis=dict(showgrid=True, gridcolor='#f0f0f0'), legend_title='Position')
fig.show()

In [7]:
# 6.6 GK Save % vs PSxG Difference (min 20 shots faced)
gk_filtered = gk_df[gk_df['shots_on_target_faced'] >= 20].copy()

fig = px.scatter(gk_filtered, x='save_pct', y='post_shot_xg_difference', color='league',
                 color_discrete_map=LEAGUE_COLORS,
                 hover_data=['player_name', 'club', 'league', 'save_pct',
                             'post_shot_xg_difference', 'shots_on_target_faced'],
                 trendline='ols', title='GK Save % vs PSxG Difference (min 20 shots faced)',
                 labels={'save_pct': 'Save %',
                         'post_shot_xg_difference': 'Post Shot xG Difference (higher = better)',
                         'league': 'League'},
                 size='shots_on_target_faced', size_max=15, opacity=0.7)
fig.add_hline(y=0, line_dash='dash', line_color='gray',
              annotation_text='xG baseline', annotation_position='right')
fig.update_layout(height=520, plot_bgcolor='white',
                  xaxis=dict(showgrid=True, gridcolor='#f0f0f0'),
                  yaxis=dict(showgrid=True, gridcolor='#f0f0f0'), legend_title='League')
fig.show()

In [8]:
# 6.7 Full Correlation Heatmap — 21 Key Stats
all_numeric_cols = [
    'goals', 'assists', 'xg', 'xag', 'shots', 'shots_on_target',
    'key_passes', 'goal_creating_actions', 'shot_creating_actions',
    'touches_opp_box', 'progressive_carries', 'progressive_passes',
    'tackles', 'interceptions', 'clearances', 'aerial_duels_won',
    'pass_completion_pct', 'dribbles_successful', 'recoveries',
    'fouls_committed', 'fouls_drawn',
]
full_corr = outfield_df[all_numeric_cols].corr().round(2)

fig = px.imshow(full_corr, text_auto=True, color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1,
                title='Full Correlation Heatmap — Key Outfield Stats (2024/25)', aspect='auto')
fig.update_layout(height=680, coloraxis_colorbar=dict(title='Correlation'))
fig.show()

print("\n✅ All 6 EDA Phases Complete!")


✅ All 6 EDA Phases Complete!
